# Week 2 — NLP Fundamentals & Error Analysis
**ComplianceGPT Lab · REU 2026 · Stony Brook University**

This is the Week 2 notebook — exercises 5 through 8, continuing from `week1_python.ipynb`.

**What you'll build this week:**
- A token counter that measures how much of a model's context window your scenarios use
- Precision, recall, and F1 score — the real metrics researchers use (accuracy alone is misleading)
- A systematic error grouping pipeline
- An oracle predicate extractor that reveals *why* predictions went wrong

Run each cell with **Shift + Enter**. Post in `#python-help` on Slack if you're stuck — include your code and the full error message.

---

In [ ]:
# Run this cell first — checks your environment
import sys
print(f"Python {sys.version}")

for pkg in ["pandas", "tiktoken", "json", "collections"]:
    try:
        __import__(pkg)
        print(f"{pkg}  ✓")
    except ImportError:
        print(f"{pkg}  NOT FOUND — run: pip install {pkg}")

print("\nSetup check complete.")

---
## Exercise 5 — Token Counting

**The concept:** LLMs don't process words — they process *tokens* (word-pieces). Every model has a context window measured in tokens. Gemma3:4B has an 8,192-token context window. If your prompt + scenario exceeds that, the model *silently truncates* — and you lose legal facts without knowing it.

**Goal:** Count tokens in real HIPAA scenarios and understand how much context window each one consumes.

**Instructions:**
1. Install tiktoken if needed: `pip install tiktoken`
2. Use the tokenizer below to count tokens in each scenario
3. Calculate: what percentage of Gemma3:4B's context window does each scenario use?
4. The prompt template in ComplianceGPT uses about 400 tokens. Which scenarios risk truncation?

**Expected output (example):**
```
Scenario 1: 87 tokens  (1.1% of context window)
Scenario 2: 312 tokens (3.8% of context window)
Scenario 3: 1,204 tokens (14.7% of context window)
⚠ Scenario 3 + prompt template = ~1,600 tokens total — safe, but monitor longer cases
```

In [ ]:
import tiktoken

# GPT-4 / Claude uses cl100k_base — a close approximation for any modern LLM
enc = tiktoken.get_encoding("cl100k_base")

CONTEXT_WINDOW = 8192   # Gemma3:4B context limit
PROMPT_OVERHEAD = 400   # approximate tokens used by the ComplianceGPT prompt template

# Three real-style HIPAA scenarios (shortened for this exercise)
scenarios = [
    # Scenario A — short, clear
    """A physician at a community health clinic disclosed a patient's HIV-positive 
    status to the patient's employer without the patient's written authorization. 
    The employer had contacted the clinic requesting information about the employee's 
    recent absences.""",

    # Scenario B — medium, judicial context
    """A hospital received a civil subpoena from an attorney representing the 
    plaintiff in a malpractice lawsuit. The subpoena requested the patient's complete 
    medical records. The hospital's legal department reviewed the subpoena and found 
    that the patient had been served with notice of the subpoena fourteen days prior, 
    that the time permitted for the patient to raise objections had passed without 
    any objection being filed, and that the requesting attorney had provided a written 
    certification to that effect. The hospital disclosed the records to the attorney.""",

    # Scenario C — long, complex multi-party situation
    """A state department of corrections requested the medical records of an inmate 
    who had been transferred from a county jail. The inmate's prior treating physician 
    at the county jail clinic received the request without any accompanying legal 
    instrument. The physician was told verbally by the corrections officer that 
    the records were needed for continuity of care. The physician consulted with 
    the clinic's privacy officer, who noted that under HIPAA, disclosures to 
    correctional institutions are permitted under 45 CFR 164.512(k)(5) when 
    the correctional institution represents that such PHI is necessary for: 
    (A) the provision of health care to the individual; (B) the health and safety 
    of the individual or other inmates; (C) the health and safety of the officers 
    or employees of or others at the correctional institution; (D) the health 
    and safety of such individuals and officers or other persons responsible for 
    the transporting of inmates or their transfer from one institution, facility, 
    or setting to another; (E) law enforcement on the premises of the correctional 
    institution; or (F) the administration and maintenance of the safety, security, 
    and good order of the correctional institution. The physician disclosed the records."""
]

# ── Exercise 5: Count tokens for each scenario ───────────────────────────────
print(f"{'Scenario':<12} {'Tokens':>8} {'% of window':>12} {'With prompt':>12} {'Truncation risk':>16}")
print("-" * 64)

for i, scenario in enumerate(scenarios, 1):
    # YOUR CODE HERE: count tokens and fill in the table
    token_count = None   # replace with: len(enc.encode(scenario))
    pct_window  = None   # replace with: token_count / CONTEXT_WINDOW * 100
    total       = None   # replace with: token_count + PROMPT_OVERHEAD
    risk        = None   # replace with: "YES" if total > CONTEXT_WINDOW * 0.9 else "no"

    print(f"Scenario {i:<3} {token_count:>8} {pct_window:>11.1f}% {total:>12} {risk:>16}")

# ── Reflection question (answer in the markdown cell below) ─────────────────
print("\nReflection: which scenario is closest to causing a truncation problem?")

**Your reflection here (double-click to edit):**

Which scenario uses the most tokens? Is any scenario at risk of being truncated by Gemma3:4B? What would happen to the compliance verdict if the key legal fact (e.g., the sentence about notification) was cut off?

_Your answer:_

---
## Exercise 6 — Precision, Recall, and F1

**Why accuracy alone is misleading for compliance:**

Imagine a model that always outputs DENIED. On a dataset with 90% DENIED ground truths, this model gets **90% accuracy** — but it's completely useless. It will block every legitimate disclosure.

We use three metrics instead:
- **Precision** — of all the PERMITTED verdicts the model gave, what fraction were actually correct?
  - Low precision = too many false permissions (PHI leaks)
- **Recall** — of all the cases that *should* have been PERMITTED, what fraction did the model catch?
  - Low recall = over-blocking legitimate disclosures
- **F1** — the harmonic mean of precision and recall; punishes imbalanced trade-offs

**Instructions:**
1. Implement `precision_recall_f1()` from scratch (no sklearn — use basic arithmetic)
2. Test it on the example data
3. Interpret what the numbers mean for a HIPAA compliance system

In [ ]:
def precision_recall_f1(predicted, ground_truth, positive_label="PERMITTED"):
    """
    Compute precision, recall, and F1 for a binary classification task.

    Args:
        predicted:      list of predicted labels
        ground_truth:   list of correct labels
        positive_label: which label is the "positive" class (default: PERMITTED)

    Returns:
        tuple: (precision, recall, f1) — each a float between 0.0 and 1.0
    """
    # Count TP, FP, FN
    # TP: predicted PERMITTED, actually PERMITTED
    # FP: predicted PERMITTED, actually DENIED  ← PHI leak risk!
    # FN: predicted DENIED, actually PERMITTED  ← over-blocking

    tp = 0
    fp = 0
    fn = 0

    for pred, gt in zip(predicted, ground_truth):
        # YOUR CODE HERE
        pass

    # YOUR CODE HERE: compute precision, recall, f1
    # Handle division by zero: if denominator is 0, return 0.0
    precision = 0.0  # replace with formula
    recall    = 0.0  # replace with formula
    f1        = 0.0  # replace with formula

    return precision, recall, f1


# ── Test 1: realistic ComplianceGPT results (20 cases) ──────────────────────
predicted_20 = [
    "PERMITTED", "DENIED",    "PERMITTED", "DENIED",    "PERMITTED",
    "DENIED",    "PERMITTED", "PERMITTED", "DENIED",    "DENIED",
    "PERMITTED", "DENIED",    "DENIED",    "PERMITTED", "DENIED",
    "PERMITTED", "DENIED",    "PERMITTED", "DENIED",    "PERMITTED"
]
ground_truth_20 = [
    "PERMITTED", "DENIED",    "DENIED",    "DENIED",    "PERMITTED",
    "PERMITTED", "PERMITTED", "DENIED",    "DENIED",    "PERMITTED",
    "PERMITTED", "DENIED",    "DENIED",    "PERMITTED", "DENIED",
    "PERMITTED", "DENIED",    "DENIED",    "DENIED",    "PERMITTED"
]

p, r, f1 = precision_recall_f1(predicted_20, ground_truth_20)
accuracy  = sum(p == g for p, g in zip(predicted_20, ground_truth_20)) / len(predicted_20)

print("Test 1 — Realistic results (20 cases)")
print(f"  Accuracy:  {accuracy:.1%}")
print(f"  Precision: {p:.1%}  (of cases we said PERMITTED, this fraction was correct)")
print(f"  Recall:    {r:.1%}  (of truly PERMITTED cases, we caught this fraction)")
print(f"  F1:        {f1:.1%}")

# ── Test 2: always-DENIED model ──────────────────────────────────────────────
predicted_denied = ["DENIED"] * 20
p2, r2, f12 = precision_recall_f1(predicted_denied, ground_truth_20)
acc2 = sum(p == g for p, g in zip(predicted_denied, ground_truth_20)) / 20

print("\nTest 2 — Always-DENIED model")
print(f"  Accuracy:  {acc2:.1%}  ← looks OK!")
print(f"  Precision: {p2:.1%}  ← undefined (no PERMITTED predictions)")
print(f"  Recall:    {r2:.1%}  ← catches ZERO legitimate permissions")
print(f"  F1:        {f12:.1%}")
print("\n→ This is why accuracy alone is misleading for HIPAA compliance.")

**Reflection (answer below):**

1. In Test 1, which metric is worst — precision or recall? What does that mean operationally? (Think: PHI leaks vs. over-blocking.)
2. In Test 2, the always-DENIED model has surprisingly high accuracy on this dataset. What's the PERMITTED/DENIED ratio in `ground_truth_20`? Does that explain it?
3. For a HIPAA compliance system used in a hospital, would you prioritize high precision or high recall? Why? (Hint: the cost of each error type is different.)

_Your answers:_

1.

2.

3.

---
## Exercise 7 — Batch Error Analysis by Category

**The concept:** Accuracy averaged over all 137 cases hides important patterns. What if the model is 100% accurate on Treatment/Payment/Operations cases, but only 50% accurate on Judicial cases? That's the finding from the ComplianceGPT paper.

**Goal:** Group errors by purpose category (TPO, law enforcement, judicial, etc.) to find where the model is struggling.

**Instructions:**
1. Load the simulated results DataFrame below (or use your real batch results if you ran them)
2. Add an `error_type` column: `"FP"`, `"FN"`, or `"correct"`
3. Group by `purpose_category` and compute: count, accuracy, FP rate, FN rate
4. Print a sorted table showing which category has the highest error rate
5. Answer the reflection question

In [ ]:
import pandas as pd

# Simulated batch results — 30 cases across 5 HIPAA exception categories
# In your real project, replace this with:
#   df = pd.read_csv("../results/yourname_week2_batch.csv")
data = {
    "scenario_id": [f"HHS-{i:03d}" for i in range(1, 31)],
    "purpose_category": [
        # Treatment/Payment/Operations — generally reliable
        "tpo", "tpo", "tpo", "tpo", "tpo", "tpo",
        # Patient Authorization — very reliable
        "authorization", "authorization", "authorization", "authorization",
        # Law Enforcement — harder (6 sub-conditions)
        "law_enforcement", "law_enforcement", "law_enforcement",
        "law_enforcement", "law_enforcement", "law_enforcement",
        # Judicial Proceedings — hardest (oracle predicate failures)
        "judicial", "judicial", "judicial", "judicial",
        "judicial", "judicial", "judicial", "judicial",
        # Research/Public Health — medium difficulty
        "research", "public_health", "research", "public_health",
        "research", "public_health",
    ],
    "ground_truth": [
        "PERMITTED", "PERMITTED", "DENIED",    "PERMITTED", "PERMITTED", "DENIED",
        "PERMITTED", "DENIED",    "PERMITTED", "PERMITTED",
        "DENIED",    "PERMITTED", "DENIED",    "DENIED",    "PERMITTED", "DENIED",
        "PERMITTED", "DENIED",    "PERMITTED", "DENIED",
        "DENIED",    "PERMITTED", "DENIED",    "PERMITTED",
        "PERMITTED", "DENIED",    "PERMITTED", "DENIED",
        "PERMITTED", "PERMITTED",
    ],
    "verdict_norm": [
        "PERMITTED", "PERMITTED", "DENIED",    "PERMITTED", "PERMITTED", "DENIED",   # tpo: all correct
        "PERMITTED", "DENIED",    "PERMITTED", "PERMITTED",                           # auth: all correct
        "DENIED",    "DENIED",    "DENIED",    "DENIED",    "PERMITTED", "DENIED",   # LE: 1 FN
        "PERMITTED", "PERMITTED", "PERMITTED", "DENIED",                             # judicial: 1 FP, 1 FN
        "DENIED",    "PERMITTED", "DENIED",    "DENIED",                             # judicial cont.: 1 FP
        "PERMITTED", "DENIED",    "PERMITTED", "DENIED",
        "PERMITTED", "PERMITTED",
    ]
}
df = pd.DataFrame(data)

# ── Step 1: Add error_type column ─────────────────────────────────────────────
# YOUR CODE HERE: add df["error_type"] with values "correct", "FP", or "FN"
# FP = model said PERMITTED, truth is DENIED  (dangerous: PHI leak)
# FN = model said DENIED, truth is PERMITTED  (over-blocking)

df["error_type"] = "correct"   # start with all correct, then overwrite

# YOUR CODE HERE:
# df.loc[condition_for_FP, "error_type"] = "FP"
# df.loc[condition_for_FN, "error_type"] = "FN"

# ── Step 2: Group by purpose_category ────────────────────────────────────────
# YOUR CODE HERE: create a summary table
# For each category, compute: total, correct, FP, FN, accuracy

summary = df.groupby("purpose_category").apply(
    lambda g: pd.Series({
        "total":    len(g),
        "correct":  (g["error_type"] == "correct").sum(),
        "FP":       (g["error_type"] == "FP").sum(),
        "FN":       (g["error_type"] == "FN").sum(),
        "accuracy": (g["error_type"] == "correct").mean(),
    })
).reset_index()

# Sort by accuracy ascending (worst categories first)
summary = summary.sort_values("accuracy")

print("Error Analysis by HIPAA Exception Category")
print("=" * 60)
print(summary.to_string(index=False, float_format="{:.1%}".format))

print(f"\nOverall accuracy: {(df['error_type'] == 'correct').mean():.1%}")
print(f"Total FP (PHI leak risk): {(df['error_type'] == 'FP').sum()}")
print(f"Total FN (over-blocking): {(df['error_type'] == 'FN').sum()}")

**Reflection:**

1. Which category has the lowest accuracy in this dataset? Does that match what the ComplianceGPT paper found?
2. What's the ratio of FP to FN errors in the judicial category? Why does that matter for security?
3. If you were going to spend one week improving the model, which category would you focus on, and why?

_Your answers:_

1.

2.

3.

---
## Exercise 8 — Oracle Predicate Analysis

**The concept:** When ComplianceGPT returns the wrong verdict, it's almost always because an oracle predicate was extracted incorrectly. Either:
- A predicate was set `True` when it shouldn't be (oracle hallucination → FP)
- A predicate was left `False` when the scenario clearly showed it was present (oracle under-extraction → FN)

This exercise teaches you to trace from a wrong verdict back to the specific oracle predicate that caused it.

**Instructions:**
1. Parse the JSON in `scenario_json` column to extract oracle predicate values
2. For FP rows only: find which oracle predicates are `True` — these are the over-fired predicates
3. Count how often each predicate appears in FP rows vs. correct rows
4. Find the "most suspicious" oracle predicate (fires most often in FP rows)

**Tip:** The oracle predicates are the keys in the JSON whose values are `True` or `False` (booleans).

In [ ]:
import json
from collections import Counter

# Simulated extraction outputs with scenario_json
# In your real project, this comes from your batch results CSV
raw_results = [
    {
        "scenario_id": "HHS-017",
        "ground_truth": "DENIED",
        "verdict_norm": "PERMITTED",    # FP — PHI leak!
        "scenario_json": json.dumps({
            "sender_role": "hospital",
            "receiver_role": "attorney",
            "purpose": "judicial",
            "has_court_order": True,                   # WRONG — no court order existed
            "has_lawful_process_with_assurance": False,
            "made_reasonable_effort_to_notify": False,
            "obtained_authorization_164_508": False,
        })
    },
    {
        "scenario_id": "HHS-021",
        "ground_truth": "DENIED",
        "verdict_norm": "PERMITTED",    # FP
        "scenario_json": json.dumps({
            "sender_role": "clinic",
            "receiver_role": "law_enforcement",
            "purpose": "law_enforcement",
            "has_court_order": True,                   # WRONG
            "is_required_by_law": True,                # WRONG
            "obtained_authorization_164_508": False,
        })
    },
    {
        "scenario_id": "HHS-019",
        "ground_truth": "PERMITTED",
        "verdict_norm": "DENIED",       # FN — over-blocking
        "scenario_json": json.dumps({
            "sender_role": "hospital",
            "receiver_role": "attorney",
            "purpose": "judicial",
            "has_court_order": False,
            "has_lawful_process_with_assurance": False,   # WRONG — assurances were provided
            "made_reasonable_effort_to_notify": False,    # WRONG — notification was made
            "obtained_authorization_164_508": False,
        })
    },
    {
        "scenario_id": "HHS-001",
        "ground_truth": "PERMITTED",
        "verdict_norm": "PERMITTED",    # correct
        "scenario_json": json.dumps({
            "sender_role": "physician",
            "receiver_role": "insurer",
            "purpose": "tpo",
            "has_court_order": False,
            "has_treatment_relationship": True,
            "obtained_authorization_164_508": False,
        })
    },
    {
        "scenario_id": "HHS-023",
        "ground_truth": "DENIED",
        "verdict_norm": "PERMITTED",    # FP
        "scenario_json": json.dumps({
            "sender_role": "hospital",
            "receiver_role": "attorney",
            "purpose": "judicial",
            "has_court_order": True,                   # WRONG again
            "has_lawful_process_with_assurance": False,
            "made_reasonable_effort_to_notify": False,
            "obtained_authorization_164_508": False,
        })
    },
]

df_results = pd.DataFrame(raw_results)

# ── Step 1: Parse oracle predicates ─────────────────────────────────────────
def parse_oracles(row):
    """Return dict of oracle predicates (boolean values only) from scenario_json."""
    try:
        sj = json.loads(row["scenario_json"])
        # YOUR CODE HERE: return only keys whose value is True or False (isinstance check)
        return {}   # replace with real extraction
    except (json.JSONDecodeError, KeyError):
        return {}

df_results["oracles"] = df_results.apply(parse_oracles, axis=1)

# ── Step 2: Label error type ─────────────────────────────────────────────────
df_results["error_type"] = "correct"
# YOUR CODE HERE: label FP and FN rows

# ── Step 3: Which oracles fire most often in FP rows? ───────────────────────
fp_rows = df_results[df_results["error_type"] == "FP"]
correct_rows = df_results[df_results["error_type"] == "correct"]

# Count which oracle predicates are True in FP rows
fp_oracle_counts = Counter()
for _, row in fp_rows.iterrows():
    for pred, val in row["oracles"].items():
        if val is True:
            fp_oracle_counts[pred] += 1

print("Oracle predicates set True in FALSE POSITIVE rows:")
print("(These are the over-fired predicates — likely hallucinations)")
print()
for pred, count in fp_oracle_counts.most_common():
    print(f"  {pred:<45} fires in {count}/{len(fp_rows)} FP rows")

print()
print(f"Total FP rows: {len(fp_rows)}")
print(f"Total FN rows: {(df_results['error_type'] == 'FN').sum()}")
print(f"Total correct: {(df_results['error_type'] == 'correct').sum()}")

**Reflection:**

1. Which oracle predicate fires most often in the FP rows? Does this match what the ComplianceGPT paper found about the judicial pathway?
2. For the FN row (HHS-019): which predicates should be `True` but are `False`? What would you add to the prompt to help the model extract these correctly?
3. Write one sentence describing the "oracle hallucination" problem in your own words — as if you were explaining it to someone who doesn't know what an oracle predicate is.

_Your answers:_

1.

2.

3.

---
## Bonus — Run on Your Real Batch Results

If you've run `batch_runner.py` this week, load your real results and apply exercises 6–8 to them.

**Instructions:**
1. Load your CSV: `df = pd.read_csv("../results/yourname_week2_batch.csv")`
2. Verify the column names match: `scenario_json`, `verdict_norm`, `ground_truth`, `match`
3. Apply `precision_recall_f1()` from Exercise 6 to your real results
4. Apply `parse_oracles()` from Exercise 8 to your real results
5. Find the single oracle predicate that fired most often in your FP rows — this is your project hypothesis

In [ ]:
import pandas as pd

# ── Load your real results ───────────────────────────────────────────────────
CSV_PATH = "../results/YOUR_NAME_week2_batch.csv"   # ← change this!

try:
    df_real = pd.read_csv(CSV_PATH)
    print(f"Loaded {len(df_real)} rows from {CSV_PATH}")
    print(f"Columns: {df_real.columns.tolist()}")
    print(f"\nFirst row:")
    print(df_real.iloc[0])

    # Apply your functions from exercises 6-8 here
    # YOUR CODE HERE

except FileNotFoundError:
    print(f"File not found: {CSV_PATH}")
    print("Run batch_runner.py first, or check the path above.")

---
## Submission

**Before Friday 5pm:**
1. Complete all 4 exercises (5–8) — fill in the `# YOUR CODE HERE` sections
2. Fill in the reflection questions under each exercise
3. Make sure all cells run without errors (`Kernel → Restart & Run All`)
4. Download as `.ipynb` and commit to your GitHub fork, OR share a Google Colab link in `#deliverables` on Slack

**Stuck?** Post in `#python-help` with your code and the full error message. Do not wait until Friday.

---
*ComplianceGPT Lab · AI Innovation & Diffusion REU 2026 · Stony Brook University*